In [1]:
%pip install -U "transformers>=4.41" "datasets>=2.20" "accelerate>=0.33" "evaluate>=0.4" sentencepiece sacrebleu tqdm ftfy unidecode langid
# optional: for progress bars in map


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, random, numpy as np, torch, inspect, re, unicodedata, langid
import evaluate
from datasets import load_dataset, load_from_disk, DatasetDict
from ftfy import fix_text
from unidecode import unidecode
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import matplotlib.pyplot as plt
import evaluate
import numpy as np
import inspect

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Device={device} | bf16={use_bf16} | fp16={use_fp16}")


Device=cuda | bf16=True | fp16=False


c:\Users\jeeva\anaconda3\Lib\site-packages\torch\__init__.py:1628: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:50.)
  _C._set_float32_matmul_precision(precision)


In [24]:
# --- Cell 1 (revised knobs for both pairs) ---
# Data sizes / lengths
MAX_SAMPLES   = 1200   # final cap *after* cleaning (per split)
EVAL_SAMPLES  = 500
MAX_SRC_LEN   = 128
MAX_TGT_LEN   = 128

# Training
BATCH       = 8
GRAD_ACCUM  = 1
MAX_STEPS   = 1000

# Cleaning — unified for all pairs (prevents EN→PT from going empty)
MIN_TOKS    = 2        # was 3
MAX_TOKS    = 512      # was 256
MAX_RATIO   = 4.0      # was 3.0
LID_CONF    = 0.75     # was 0.90 (Portuguese often <0.9)

# Domain filter OFF while you debug emptiness; turn on later if you want.
DOMAIN_FILTER = False


In [25]:
# --- Cell 2: unified cleaning helpers (applies to ALL pairs equally) ---
import re, unicodedata, langid
from ftfy import fix_text
from datasets import load_dataset, DatasetDict

_WS_RE = re.compile(r"\s+")
_DASHES = dict.fromkeys(map(ord, "–—−"), "-")

def normalize_text(s: str) -> str:
    if s is None: return ""
    s = fix_text(s)
    s = s.translate(_DASHES)
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u00A0", " ")
    s = s.replace("’","'").replace("“",'"').replace("”",'"')
    s = re.sub(r"-\s*\n\s*", "", s)
    s = s.replace("\n", " ")
    s = _WS_RE.sub(" ", s).strip()
    return s

# light garbage filter (urls/templates/entities)
_HTML_RE = re.compile(r"\{?\{.*\}\}?|<[^>]{1,120}>|&[a-z]{2,10};", re.I)
def looks_garbage(s: str) -> bool:
    u = s.lower()
    if u.count("http://") + u.count("https://") + u.count("www.") > 2:
        return True
    return bool(_HTML_RE.search(s))

def _length_ok(ex, min_t=MIN_TOKS, max_t=MAX_TOKS, max_ratio=MAX_RATIO):
    ls, lt = len(ex["src"].split()), len(ex["tgt"].split())
    if not (min_t <= ls <= max_t and min_t <= lt <= max_t):
        return False
    return max(ls, lt) / max(1, min(ls, lt)) <= max_ratio

def _lid_ok(ex, src_lang, tgt_lang, thr=LID_CONF):
    s_lang, s_conf = langid.classify(ex["src"])
    t_lang, t_conf = langid.classify(ex["tgt"])
    # keep if correct and confident; drop only when confidently wrong
    return ((s_lang == src_lang and s_conf >= thr) or (s_conf < thr)) and \
           ((t_lang == tgt_lang and t_conf >= thr) or (t_conf < thr))

def _dedup_pairs(ds):
    seen, keep = set(), []
    for i in range(len(ds)):
        key = (ds[i]["src"], ds[i]["tgt"])
        if key in seen: 
            continue
        seen.add(key); keep.append(i)
    return ds.select(keep)

def _maybe_domain_filter(ds):
    if not DOMAIN_FILTER:
        return ds
    sci_kw = [
        "protein","molecule","polymerase","confidence interval","experiment","dataset",
        "algorithm","neural","equation","statistical","model","clinical","biomedical",
        "cell","significant","regression","trial","citation","figure","table","RNA",
        "DNA","enzyme","vector","matrix"
    ]
    rex = re.compile("|".join(map(re.escape, sci_kw)), re.I)
    return ds.filter(lambda ex: bool(rex.search(ex["src"])) or bool(rex.search(ex["tgt"])))

def build_clean_pair(pair: str, max_samples: int | None):
    """One path for all pairs: shuffle→normalize→garbage→LID(~0.75)→len/ratio(4.0)→pair-dedup→cap-after."""
    src_lang, tgt_lang = pair.split("-")
    langid.set_languages([src_lang, tgt_lang])
    raw = DatasetDict({
        "train":      load_dataset("opus100", pair, split="train").shuffle(SEED),
        "validation": load_dataset("opus100", pair, split="validation").shuffle(SEED),
    })

    def to_cols(b):
        tr = b["translation"]
        return {"src": normalize_text(tr[src_lang]), "tgt": normalize_text(tr[tgt_lang])}

    d_train = raw["train"].map(to_cols, remove_columns=raw["train"].column_names, desc=f"normalize {pair}/train")
    d_valid = raw["validation"].map(to_cols, remove_columns=raw["validation"].column_names, desc=f"normalize {pair}/valid")

    d_train = d_train.filter(lambda ex: not looks_garbage(ex["src"]) and not looks_garbage(ex["tgt"]),
                             desc=f"garbage {pair}/train")
    d_valid = d_valid.filter(lambda ex: not looks_garbage(ex["src"]) and not looks_garbage(ex["tgt"]),
                             desc=f"garbage {pair}/valid")

    d_train = d_train.filter(lambda ex: _lid_ok(ex, src_lang, tgt_lang, LID_CONF), desc=f"LID {pair}/train")
    d_valid = d_valid.filter(lambda ex: _lid_ok(ex, src_lang, tgt_lang, LID_CONF), desc=f"LID {pair}/valid")

    d_train = d_train.filter(_length_ok, desc=f"len/ratio {pair}/train")
    d_valid = d_valid.filter(_length_ok, desc=f"len/ratio {pair}/valid")

    d_train = _dedup_pairs(d_train)   # pair-only dedup (do NOT dedup by src)
    d_valid = _dedup_pairs(d_valid)

    d_train = _maybe_domain_filter(d_train)
    d_valid = _maybe_domain_filter(d_valid)

    # Cap AFTER filtering so we don't slice a bad block
    if max_samples:
        d_train = d_train.select(range(min(max_samples, len(d_train))))
        d_valid = d_valid.select(range(min(EVAL_SAMPLES, len(d_valid))))

    # rename for downstream
    def to_expected(b): return {"src_text": b["src"], "tgt_text": b["tgt"]}
    train = d_train.map(to_expected, remove_columns=d_train.column_names)
    valid = d_valid.map(to_expected, remove_columns=d_valid.column_names)
    return train, valid

def dataset_summary(name, ds):
    print(f"{name}: {len(ds)} rows | cols={ds.column_names}")
    if len(ds):
        ex = ds[0]
        print("  sample:", ex["src_text"][:60], "→", ex["tgt_text"][:60])


In [ ]:
# # ===== Sanity check + auto-relax for EN→PT if filters nuked the data =====

# def dataset_summary(name, ds):
#     try:
#         print(f"{name}: {len(ds)} rows | cols={ds.column_names[:6]}{'...' if len(ds.column_names)>6 else ''}")
#         if len(ds): 
#             ex = ds[0]
#             print(f"  sample-> src_text[:60]={ex['src_text'][:60]!r} | tgt_text[:60]={ex['tgt_text'][:60]!r}")
#     except Exception as e:
#         print(f"{name}: <error reading> {e}")

# dataset_summary("EN→ES train", train_es)
# dataset_summary("EN→ES valid", valid_es)
# dataset_summary("EN→PT train", train_pt)
# dataset_summary("EN→PT valid", valid_pt)

# if len(train_pt) == 0 or len(valid_pt) == 0:
#     print("\n[WARN] EN→PT split is empty after cleaning. Rebuilding with relaxed filters…")

#     import re, langid, unicodedata
#     from datasets import load_dataset, DatasetDict
#     from ftfy import fix_text

#     # reuse your normalizer (lightweight)
#     WS_RE = re.compile(r"\s+")
#     DASHES = dict.fromkeys(map(ord, "–—−"), "-")
#     def normalize_text_relaxed(s: str) -> str:
#         if s is None: return ""
#         s = fix_text(s)
#         s = s.translate(DASHES)
#         s = unicodedata.normalize("NFC", s)
#         s = s.replace("\u00A0", " ")
#         s = s.replace("’","'").replace("“",'"').replace("”",'"')
#         s = re.sub(r"-\s*\n\s*", "", s)
#         s = s.replace("\n", " ")
#         s = WS_RE.sub(" ", s).strip()
#         return s

#     def clean_pair_relaxed(pair: str, max_samples: int = None):
#         src_lang, tgt_lang = pair.split("-")
#         langid.set_languages([src_lang, tgt_lang])
#         raw = DatasetDict({
#             "train":      load_dataset("opus100", pair, split="train"),
#             "validation": load_dataset("opus100", pair, split="validation"),
#         })
#         # light pre-cap
#         if max_samples:
#             raw["train"]      = raw["train"].select(range(min(max_samples*2, len(raw["train"]))))
#             raw["validation"] = raw["validation"].select(range(min(max_samples,   len(raw["validation"]))))

#         # normalize
#         def to_cols(b):
#             tr = b["translation"]
#             return {"src": normalize_text_relaxed(tr[src_lang]), "tgt": normalize_text_relaxed(tr[tgt_lang])}
#         norm = {sp: raw[sp].map(to_cols, remove_columns=raw[sp].column_names) for sp in raw}

#         # relaxed LID (allow lower confidence + fallback keep if uncertain)
#         LID_CONF_RELAX = 0.60
#         def lid_ok(ex):
#             s_lang, s_conf = langid.classify(ex["src"])
#             t_lang, t_conf = langid.classify(ex["tgt"])
#             s_good = (s_lang == src_lang and s_conf >= LID_CONF_RELAX) or (s_conf < LID_CONF_RELAX)
#             t_good = (t_lang == tgt_lang and t_conf >= LID_CONF_RELAX) or (t_conf < LID_CONF_RELAX)
#             return s_good and t_good
#         lid = {sp: norm[sp].filter(lid_ok) for sp in norm}

#         # relaxed length + ratio
#         MIN_TOKS_R, MAX_TOKS_R, MAX_RATIO_R = 2, 512, 4.0
#         def length_ok(ex):
#             ls, lt = len(ex["src"].split()), len(ex["tgt"].split())
#             if not (MIN_TOKS_R <= ls <= MAX_TOKS_R and MIN_TOKS_R <= lt <= MAX_TOKS_R):
#                 return False
#             r = max(ls, lt) / max(1, min(ls, lt))
#             return r <= MAX_RATIO_R
#         filt = {sp: lid[sp].filter(length_ok) for sp in lid}

#         # dedupe by exact pairs only (do NOT dedupe by src to avoid over-pruning)
#       def dedup_pairs(ds):
#             seen, keep = set(), []
#             for i in range(len(ds)):
#             key = (ds[i]["src"], ds[i]["tgt"])
#             if key in seen: continue
#             seen.add(key); keep.append(i)
#             return ds.select(keep)

#         # final cap
#         if max_samples:
#             cleaned["train"]      = cleaned["train"].select(range(min(max_samples,   len(cleaned["train"]))))
#             cleaned["validation"] = cleaned["validation"].select(range(min(EVAL_SAMPLES, len(cleaned["validation"]))))

#         # to expected cols
#         def to_expected(b): return {"src_text": b["src"], "tgt_text": b["tgt"]}
#         train = cleaned["train"].map(to_expected, remove_columns=cleaned["train"].column_names)
#         valid = cleaned["validation"].map(to_expected, remove_columns=cleaned["validation"].column_names)
#         return train, valid

#     # rebuild EN→PT with relaxed constraints
#     train_pt, valid_pt = clean_pair_relaxed("en-pt", max_samples=MAX_SAMPLES)

#     dataset_summary("EN→PT(train, relaxed)", train_pt)
#     dataset_summary("EN→PT(valid, relaxed)", valid_pt)

#     if len(train_pt) == 0:
#         raise RuntimeError("EN→PT still empty after relaxed cleaning. Consider turning off LID or broadening filters further.")


In [ ]:
# # --- Clean helpers & builder (drop-in replacement) ---

# import re, unicodedata, langid
# from datasets import load_dataset, DatasetDict
# from ftfy import fix_text

# # Make sure these constants are defined somewhere above:
# # MIN_TOKS = 3; MAX_TOKS = 256; MAX_RATIO = 3.0; LID_CONF = 0.90; EVAL_SAMPLES = 500

# WS_RE = re.compile(r"\s+")
# DASHES = dict.fromkeys(map(ord, "–—−"), "-")  # normalize dash variants to "-"

# def normalize_text(s: str) -> str:
#     if s is None:
#         return ""
#     s = fix_text(s)
#     s = s.translate(DASHES)
#     s = unicodedata.normalize("NFC", s)
#     s = s.replace("\u00A0", " ")                  # NBSP → space
#     s = s.replace("’","'").replace("“",'"').replace("”",'"')
#     s = re.sub(r"-\s*\n\s*", "", s)               # de-hyphenate at line breaks
#     s = s.replace("\n", " ")
#     s = WS_RE.sub(" ", s).strip()
#     return 

# def deduplicate_pair_and_src(ds):
#     """
#     Keep first occurrence of each (src,tgt) AND first occurrence of each src.
#     No temp columns; version-agnostic.
#     """
#     seen_pairs, seen_src, keep_idx = set(), set(), []
#     for i in range(len(ds)):
#         ex = ds[i]
#         s, t = ex["src"], ex["tgt"]
#         pair = (s, t)
#         if pair in seen_pairs or s in seen_src:
#             continue
#         seen_pairs.add(pair)
#         seen_src.add(s)
#         keep_idx.append(i)
#     return ds.select(keep_idx)

# def build_clean_dataset(pair: str, save_dir: str, max_samples: int = None, domain_filter: bool = True):
#     """
#     Load OPUS100 pair, clean, (optionally) domain-filter, downselect, save.
#     Returns DatasetDict with 'train' and 'validation'.
#     """
#     src_lang, tgt_lang = pair.split("-")
#     langid.set_languages([src_lang, tgt_lang])

#     # 1) Load
#     raw = DatasetDict({
#         "train":      load_dataset("opus100", pair, split="train"),
#         "validation": load_dataset("opus100", pair, split="validation"),
#     })
#     # (Optional) cap upfront for speed
#     if max_samples:
#         raw["train"] = raw["train"].select(range(min(max_samples * 2, len(raw["train"]))))
#         raw["validation"] = raw["validation"].select(range(min(max_samples, len(raw["validation"]))))

#     # 2) Normalize
#     def to_src_tgt(b):
#         tr = b["translation"]
#         return {"src": normalize_text(tr[src_lang]), "tgt": normalize_text(tr[tgt_lang])}
#     norm = {sp: raw[sp].map(to_src_tgt, remove_columns=raw[sp].column_names, desc=f"normalize {pair}/{sp}") for sp in raw}

#     # 3) Language-ID filter
#     def lid_ok(ex):
#         s_lang, s_conf = langid.classify(ex["src"])
#         t_lang, t_conf = langid.classify(ex["tgt"])
#         return (s_lang == src_lang and s_conf >= LID_CONF and t_lang == tgt_lang and t_conf >= LID_CONF)
#     lid = {sp: norm[sp].filter(lid_ok, desc=f"LID {pair}/{sp}") for sp in norm}

#     # 4) Length & ratio filter
#     def tok_count(s): return len(s.split())
#     def length_ok(ex):
#         ls, lt = tok_count(ex["src"]), tok_count(ex["tgt"])
#         if not (MIN_TOKS <= ls <= MAX_TOKS and MIN_TOKS <= lt <= MAX_TOKS):
#             return False
#         r = max(ls, lt) / max(1, min(ls, lt))
#         return r <= MAX_RATIO
#     filt = {sp: lid[sp].filter(length_ok, desc=f"length/ratio {pair}/{sp}") for sp in lid}

#     # 5) Deduplicate (exact pair + src-side)
#     dedup = {sp: deduplicate_pair_and_src(filt[sp]) for sp in filt}

#     # 6) (Optional) science-ish domain filter, with safe fallback
#     use_ds = dedup
#     if domain_filter:
#         SCI_KW = [
#             "protein","molecule","polymerase","confidence interval","experiment","dataset","algorithm",
#             "neural","translation","equation","statistical","model","clinical","biomedical","cell",
#             "significant","regression","trial","citation","figure","table","RNA","DNA","enzyme","vector","matrix"
#         ]
#         SCI_RE = re.compile("|".join([re.escape(k) for k in SCI_KW]), re.I)
#         def sci_like(ex): return bool(SCI_RE.search(ex["src"])) or bool(SCI_RE.search(ex["tgt"]))
#         domain = {sp: dedup[sp].filter(sci_like, desc=f"domain {pair}/{sp}") for sp in dedup}
#         # fallback if a split gets emptied
#         for sp in domain:
#             if len(domain[sp]) == 0 and len(dedup[sp]) > 0:
#                 print(f"[WARN] Domain filter emptied '{pair}/{sp}'. Falling back to non-domain data.")
#                 domain[sp] = dedup[sp]
#         use_ds = domain

#     # 7) Final select sizes
#     if max_samples:
#         use_ds["train"] = use_ds["train"].select(range(min(max_samples, len(use_ds["train"]))))
#         use_ds["validation"] = use_ds["validation"].select(range(min(EVAL_SAMPLES, len(use_ds["validation"]))))

#     cleaned = DatasetDict(use_ds)
#     cleaned.save_to_disk(save_dir)
#     print(f"Saved cleaned {pair} to {save_dir}; sizes:", {k: len(v) for k, v in cleaned.items()})
#     return cleaned


In [26]:
# --- Cell 3: build both pairs using the SAME rules (no per-language fallback) ---
train_es, valid_es = build_clean_pair("en-es", MAX_SAMPLES)
train_pt, valid_pt = build_clean_pair("en-pt", MAX_SAMPLES)

dataset_summary("EN→ES train", train_es)
dataset_summary("EN→ES valid", valid_es)
dataset_summary("EN→PT train", train_pt)
dataset_summary("EN→PT valid", valid_pt)

# Hard guard so you notice early if something still wipes a split
assert len(train_es) > 0 and len(valid_es) > 0, "EN→ES is empty after cleaning."
assert len(train_pt) > 0 and len(valid_pt) > 0, "EN→PT is empty after cleaning."


normalize en-es/train:   0%|          | 0/1000000 [00:00<?, ? examples/s]

normalize en-es/valid:   0%|          | 0/2000 [00:00<?, ? examples/s]

garbage en-es/train:   0%|          | 0/1000000 [00:00<?, ? examples/s]

garbage en-es/valid:   0%|          | 0/2000 [00:00<?, ? examples/s]

LID en-es/train:   0%|          | 0/999493 [00:00<?, ? examples/s]

...

normalize en-pt/valid:   0%|          | 0/2000 [00:00<?, ? examples/s]

garbage en-pt/train:   0%|          | 0/1000000 [00:00<?, ? examples/s]

garbage en-pt/valid:   0%|          | 0/2000 [00:00<?, ? examples/s]

LID en-pt/train:   0%|          | 0/999666 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [20]:
tok_es = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es", use_fast=True)
mod_es = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(device)

tok_pt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-tc-big-en-pt", use_fast=True)
mod_pt = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-tc-big-en-pt").to(device)

assert getattr(mod_pt.config, "model_type", "") == "marian", "Unexpected PT model type"
print("✅ Loaded Marian models (EN→ES, EN→PT)")


✅ Loaded Marian models (EN→ES, EN→PT)


In [21]:
USE_GLOSSARY = True
GLOSSARY = {
    "polymerase chain reaction": {"es": "reacción en cadena de la polimerasa", "pt": "reação em cadeia da polimerase"},
    "confidence interval": {"es": "intervalo de confianza", "pt": "intervalo de confiança"},
}

def apply_glossary(text: str, lang: str) -> str:
    out = text
    for en_term, mapping in GLOSSARY.items():
        if lang in mapping:
            out = out.replace(en_term, mapping[lang])
    return out

def preprocess_any(batch, tok, tgt_short: str):
    targets = batch["tgt_text"]
    if USE_GLOSSARY:
        targets = [apply_glossary(t, tgt_short) for t in targets]

    model_inputs = tok(
        batch["src_text"], truncation=True, max_length=MAX_SRC_LEN,
    )
    try:
        labels = tok(
            text_target=targets, truncation=True, max_length=MAX_TGT_LEN,
        )
    except TypeError:
        from contextlib import contextmanager
        cm = tok.as_target_tokenizer() if hasattr(tok, "as_target_tokenizer") else contextmanager(lambda: (yield))()
        with cm:
            labels = tok(targets, truncation=True, max_length=MAX_TGT_LEN)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_es_tok = train_es.map(lambda b: preprocess_any(b, tok_es, "es"), batched=True, remove_columns=train_es.column_names)
valid_es_tok = valid_es.map(lambda b: preprocess_any(b, tok_es, "es"), batched=True, remove_columns=valid_es.column_names)

train_pt_tok = train_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"), batched=True, remove_columns=train_pt.column_names)
valid_pt_tok = valid_pt.map(lambda b: preprocess_any(b, tok_pt, "pt"), batched=True, remove_columns=valid_pt.column_names)

print("✅ Tokenization done (ES & PT)")


Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

✅ Tokenization done (ES & PT)


In [22]:
# ====== VERSION-COMPATIBLE TRAINING BLOCK ======
import inspect
import numpy as np
import evaluate
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)

# --- Build args safely against whatever HF version you have ---
def make_s2s_args(**wanted):
    """
    Create Seq2SeqTrainingArguments while staying compatible with older/newer
    Transformers versions: rename or drop keys that aren't supported.
    """
    sig = inspect.signature(Seq2SeqTrainingArguments.__init__)
    keys = set(sig.parameters.keys())
    w = dict(wanted)

    # 1) evaluation_strategy compatibility
    if "evaluation_strategy" in w and "evaluation_strategy" not in keys:
        if "eval_strategy" in keys:
            w["eval_strategy"] = w.pop("evaluation_strategy")
        else:
            # very old HF used evaluate_during_training=True
            w.pop("evaluation_strategy")
            if "evaluate_during_training" in keys and w.get("eval_steps", None):
                w["evaluate_during_training"] = True

    # 2) predict_with_generate compatibility
    if "predict_with_generate" in w and "predict_with_generate" not in keys:
        w.pop("predict_with_generate")

    # 3) generation_* compatibility
    for k in ["generation_num_beams", "generation_max_length"]:
        if k in w and k not in keys:
            w.pop(k)

    # 4) other sometimes-missing keys
    for k in ["save_strategy", "save_steps", "save_total_limit",
              "load_best_model_at_end", "metric_for_best_model",
              "greater_is_better", "label_smoothing_factor",
              "warmup_ratio", "lr_scheduler_type",
              "dataloader_pin_memory", "dataloader_num_workers",
              "report_to", "bf16", "fp16", "no_cuda",
              "logging_steps", "max_steps", "gradient_accumulation_steps",
              "per_device_train_batch_size", "output_dir", "learning_rate",
              "eval_steps"]:
        # keep if supported; drop if not
        if k in w and k not in keys:
            w.pop(k)

    # Show what we dropped so you know what's unsupported locally
    dropped = sorted(set(wanted) - set(w.keys()))
    if dropped:
        print("Dropped unsupported training args:", dropped)

    return Seq2SeqTrainingArguments(**w)

# --- Collators ---
collator_es = DataCollatorForSeq2Seq(tokenizer=tok_es, model=mod_es, padding="longest")
collator_pt = DataCollatorForSeq2Seq(tokenizer=tok_pt, model=mod_pt, padding="longest")

# --- Metrics on generated text (Seq2SeqTrainer will pass generated ids if supported) ---
bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

def compute_metrics_for(tok):
    def _inner(eval_pred):
        preds = getattr(eval_pred, "predictions", eval_pred[0])
        labels = getattr(eval_pred, "label_ids",    eval_pred[1])

        # unwrap tuples
        if isinstance(preds, tuple):  preds  = preds[0]
        if isinstance(labels, tuple): labels = labels[0]

        # Replace -100 with pad id before decoding
        labels = np.where(labels == -100, tok.pad_token_id, labels)

        # Decode token ids → strings
        preds_dec = tok.batch_decode(preds,  skip_special_tokens=True)
        refs_dec  = tok.batch_decode(labels, skip_special_tokens=True)

        refs_ll = [[r] for r in refs_dec]
        return {
            "bleu": bleu_metric.compute(predictions=preds_dec, references=refs_ll)["score"],
            "chrf": chrf_metric.compute(predictions=preds_dec, references=refs_ll)["score"],
        }
    return _inner

# ===== EN→ES =====
args_es = make_s2s_args(
    output_dir=f"{PROJECT_DIR}/ckpt_es",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
     remove_unused_columns=False,

    # generation during eval (will be dropped if unsupported)
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=MAX_TGT_LEN,

    # validation/checkpointing (auto-mapped/dropped as needed)
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,

    # regularization
    label_smoothing_factor=0.1,
)

trainer_es = Seq2SeqTrainer(
    model=mod_es,
    args=args_es,
    train_dataset=train_es_tok,
    eval_dataset=valid_es_tok,
    data_collator=collator_es,
    tokenizer=tok_es,
    compute_metrics=compute_metrics_for(tok_es),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("✅ EN→ES trainer ready. Starting training…")
train_output_es = trainer_es.train()
print("✅ EN→ES done.")
try:
    trainer_es.save_model(f"{PROJECT_DIR}/ckpt_es_best")
    tok_es.save_pretrained(f"{PROJECT_DIR}/ckpt_es_best")
except Exception as e:
    print("[WARN] Save best ES model skipped:", e)

# ===== EN→PT =====
args_pt = make_s2s_args(
    output_dir=f"{PROJECT_DIR}/ckpt_pt",
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    learning_rate=3e-5,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none",
    bf16=use_bf16,
    fp16=use_fp16,
    no_cuda=(device != "cuda"),
    dataloader_num_workers=0,
    dataloader_pin_memory=(device == "cuda"),
     remove_unused_columns=False,

    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=MAX_TGT_LEN,

    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_bleu",
    greater_is_better=True,

    label_smoothing_factor=0.1,
)

trainer_pt = Seq2SeqTrainer(
    model=mod_pt,
    args=args_pt,
    train_dataset=train_pt_tok,
    eval_dataset=valid_pt_tok,
    data_collator=collator_pt,
    tokenizer=tok_pt,
    compute_metrics=compute_metrics_for(tok_pt),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("✅ EN→PT trainer ready. Starting training…")
train_output_pt = trainer_pt.train()
print("✅ EN→PT done.")
try:
    trainer_pt.save_model(f"{PROJECT_DIR}/ckpt_pt_best")
    tok_pt.save_pretrained(f"{PROJECT_DIR}/ckpt_pt_best")
except Exception as e:
    print("[WARN] Save best PT model skipped:", e)
# ====== END TRAINING BLOCK ======


C:\Users\jeeva\AppData\Local\Temp\ipykernel_38384\3026738732.py:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_es = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Dropped unsupported training args: ['evaluation_strategy']
✅ EN→ES trainer ready. Starting training…


Step,Training Loss,Validation Loss,Bleu,Chrf
200,1.442700,2.185024,65.803701,85.428836
400,1.438900,2.329814,65.803701,85.428836
600,1.437700,2.354085,65.803701,85.428836
800,1.437000,2.406028,65.803701,85.428836


c:\Users\jeeva\anaconda3\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[65000]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


✅ EN→ES done.
Dropped unsupported training args: ['evaluation_strategy']


C:\Users\jeeva\AppData\Local\Temp\ipykernel_38384\3026738732.py:186: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer_pt = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


✅ EN→PT trainer ready. Starting training…


Step,Training Loss,Validation Loss,Bleu,Chrf
200,2.065700,2.608005,39.931252,62.814147
400,1.840800,2.669297,38.832594,62.215037
600,1.689900,2.727000,39.221789,62.288401
800,1.588300,2.777925,38.370431,61.749379


c:\Users\jeeva\anaconda3\Lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 4, 'bad_words_ids': [[54775]], 'forced_eos_token_id': 44670}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


✅ EN→PT done.


In [23]:
EVAL_LIMIT   = 500
MAX_NEW_TOK  = 64
MIN_NEW_TOK  = 3
NUM_BEAMS    = 4
BATCH_SIZE   = 32

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")

@torch.inference_mode()
def batch_generate(model, tok, texts, max_len_src=MAX_SRC_LEN, max_new=MAX_NEW_TOK, beams=NUM_BEAMS, batch_size=BATCH_SIZE):
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len_src)
    input_ids = enc["input_ids"].to(model.device)
    attn = enc.get("attention_mask")
    if attn is not None:
        attn = attn.to(model.device)
    preds = []
    for i in range(0, input_ids.size(0), batch_size):
        sl = slice(i, i + batch_size)
        batch = {"input_ids": input_ids[sl]}
        if attn is not None:
            batch["attention_mask"] = attn[sl]
        out = model.generate(
            **batch, max_new_tokens=max_new, min_new_tokens=MIN_NEW_TOK,
            num_beams=beams, no_repeat_ngram_size=3, length_penalty=1.0, do_sample=False,
        )
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return [p.strip() for p in preds]

def evaluate_pair(model, tok, valid_ds, lang_name="ES"):
    src = [r["src_text"] for r in valid_ds][:EVAL_LIMIT]
    refs = [r["tgt_text"] for r in valid_ds][:EVAL_LIMIT]
    preds = batch_generate(model, tok, src)

    refs_bleu = [[r] for r in refs]
    bleu = bleu_metric.compute(predictions=preds, references=refs_bleu)["score"]
    chrf = chrf_metric.compute(predictions=preds, references=refs_bleu)["score"]
    exact = np.mean([int(p == r) for p, r in zip(preds, refs)]) * 100.0

    k = min(3, len(preds))
    print(f"\n--- {lang_name} samples ---")
    for i in range(k):
        print("SRC:", src[i]); print("PRD:", preds[i]); print("REF:", refs[i]); print("---")

    print(f"{lang_name} -> Metrics on {len(preds)} examples:")
    print(f"  BLEU : {bleu:.2f}")
    print(f"  chrF : {chrf:.2f}")
    print(f"  Exact-match accuracy : {exact:.2f}%")
    return {"bleu": bleu, "chrf": chrf, "exact_acc": exact}

results = {}
results["es"] = evaluate_pair(mod_es, tok_es, valid_es, lang_name="EN→ES")
results["pt"] = evaluate_pair(mod_pt, tok_pt, valid_pt, lang_name="EN→PT")
results



--- EN→ES samples ---
SRC: OJ L 153, 18.06.2010, p. 1.
PRD: DO L 153 de 18.06.2010, p. 1.
REF: DO L 153 de 18.6.2010, p. 1.
---
EN→ES -> Metrics on 1 examples:
  BLEU : 65.80
  chrF : 85.43
  Exact-match accuracy : 0.00%

--- EN→PT samples ---
SRC: He's never done anything.
PRD: Ele nunca fez nada.
REF: Nunca fez nada.
---
SRC: 50/50 shot?
PRD: Um tiro de 50/50?
REF: 50/50 de hipóteses?
---
SRC: Yeah, and I'm letting you get your come-up. But you gonna be good?
PRD: Sim, e vou deixar que arranjes a tua ideia, mas vais ficar bem?
REF: Deixo que os enfrentes, mas vais portar-te bem?
---
EN→PT -> Metrics on 500 examples:
  BLEU : 38.90
  chrF : 61.42
  Exact-match accuracy : 9.00%


{'es': {'bleu': 65.80370064762461,
  'chrf': 85.42883627590109,
  'exact_acc': np.float64(0.0)},
 'pt': {'bleu': 38.90285157228241,
  'chrf': 61.420669534390896,
  'exact_acc': np.float64(9.0)}}

In [ ]:
def plot_loss(trainer, title):
    loss_points = [(e["step"], e["loss"]) for e in getattr(trainer.state, "log_history", []) if "loss" in e]
    if not loss_points:
        print(f"{title}: no loss logs found."); return
    steps, losses = zip(*loss_points)
    plt.figure(); plt.plot(steps, losses)
    plt.xlabel("Step"); plt.ylabel("Training loss"); plt.title(title); plt.show()

plot_loss(trainer_es, "EN→ES: Training loss")
plot_loss(trainer_pt, "EN→PT: Training loss")

def translate_es(text, max_new_tokens=128, num_beams=5):
    enc = tok_es([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_es.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_es.batch_decode(out, skip_special_tokens=True)[0]

def translate_pt(text, max_new_tokens=128, num_beams=5):
    enc = tok_pt([text], return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = mod_pt.generate(**enc, num_beams=num_beams, max_new_tokens=max_new_tokens, no_repeat_ngram_size=3, early_stopping=True)
    return tok_pt.batch_decode(out, skip_special_tokens=True)[0]
